In [0]:
%run "/Workspace/trial/trial/Tutorial"

# DELTA LAKE

In [0]:
df_sales.write.format('parquet')\
        .mode('append')\
        .option('path','abfss://destination@datalakegautam.dfs.core.windows.net/sales')\
        .save()

## Managed VS External Delta Tables

**Database**

In [0]:
%sql
CREATE DATABASE salesDB;

**Managed Table**

In [0]:
%sql
CREATE TABLE salesDB.mantable  
(
  id INT,
  name STRING,
  marks INT
)
USING DELTA  

In [0]:
%sql
INSERT INTO salesDB.mantable 
VALUES
(1,'aa',30),
(2,'bb',33),
(3,'cc',35),
(4,'DD',40)

In [0]:
%sql 
select * from salesDB.mantable;


In [0]:
%sql
DROP TABLE salesDB.mantable;

**External Table**

In [0]:
%sql
CREATE TABLE salesDB.exttable  
(
  id INT,
  name STRING,
  marks INT 
)
USING DELTA    
LOCATION 'abfss://destination@datalakegautam.dfs.core.windows.net/salesDB/exttable' 

In [0]:
%sql
INSERT INTO salesDB.exttable 
VALUES
(1,'aa',30),
(2,'bb',33),
(3,'cc',35),
(4,'DD',40)

In [0]:
%sql 
select * from salesDB.exttable;

## Delta Tables Functionalities

**INSERT**

In [0]:
%sql
INSERT INTO salesDB.exttable 
VALUES
(5,'aa',30),
(6,'bb',33),
(7,'cc',35),
(8,'DD',40)

In [0]:
%sql
select * from salesdb.exttable

**DELETE**

In [0]:
%sql
DELETE FROM salesdb.exttable 
WHERE id = 8

In [0]:
%sql
select * from salesdb.exttable

**DATA VERSIONING**

**TIME TRAVEL**

In [0]:
%sql
RESTORE TABLE salesdb.exttable TO VERSION AS OF 2;

In [0]:
%sql
select * from salesDB.exttable

**VACUUM**

**VACUUM RETAIN 0 HRS**

In [0]:
%sql
VACUUM salesdb.exttable RETAIN 0 HOURS; 

### DELTA Table Optimization

**OPTIMIZE**

In [0]:
%sql
select * from salesdb.exttable

**ZORDER BY**

In [0]:
%sql
OPTIMIZE salesdb.exttable ZORDER BY (id)

In [0]:
%sql
select * from salesdb.exttable

### AUTO LOADER

**Streaming Dataframe**

In [0]:
df = spark.readStream.format('cloudFiles')\
        .option('cloudFiles.format','parquet')\
        .option('cloudFiles.schemaLocation','abfss://aldestination@datalakegautam.dfs.core.windows.net/checkpoint')\
        .load('abfss://alsource@datalakegautam.dfs.core.windows.net')   

In [0]:
df.writeStream.format('delta')\
               .option('checkpointLocation','abfss://aldestination@datalakegautam.dfs.core.windows.net/checkpoint')\
               .option('mergeSchema','true')\
               .trigger(processingTime='5 seconds')\
               .start('abfss://aldestination@datalakegautam.dfs.core.windows.net/data')